In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory // 1024**3, 'GB')


True
Tesla T4
14 GB


In [2]:
!pip install -q einops pandas numpy matplotlib scikit-learn

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/inference_engine'
for subdir in ['data','checkpoints','weights','kernels','results']:
    os.makedirs(f'{PROJECT_DIR}/{subdir}', exist_ok=True)
print('Project dir ready:', PROJECT_DIR)


Mounted at /content/drive
Project dir ready: /content/drive/MyDrive/inference_engine


In [4]:
!git clone --quiet https://github.com/zhouhaoyi/ETDataset.git /content/ETDataset
import shutil
shutil.copy('/content/ETDataset/ETT-small/ETTh1.csv', f'{PROJECT_DIR}/data/ETTh1.csv')

import pandas as pd
df = pd.read_csv(f'{PROJECT_DIR}/data/ETTh1.csv')
print(df.shape)
print(df.columns.tolist())
print(df.head(2))


(17420, 8)
['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
                  date   HUFL   HULL   MUFL   MULL   LUFL   LULL         OT
0  2016-07-01 00:00:00  5.827  2.009  1.599  0.462  4.203  1.340  30.531000
1  2016-07-01 01:00:00  5.693  2.076  1.492  0.426  4.142  1.371  27.787001


In [8]:
import numpy as np
np.random.seed(42)
df_anom = df.copy()
labels  = np.zeros(len(df), dtype=int)

# Point spikes: sudden large deviation
spike_idx = np.random.choice(len(df), size=100, replace=False)
df_anom.loc[spike_idx,'OT'] += np.random.choice([-1,1],100) * df['OT'].std() * 4
labels[spike_idx] = 1

# Level shifts: sustained offset over a window
for _ in range(6):
    s = np.random.randint(1000, len(df)-200)
    l = np.random.randint(48, 192)
    df_anom.loc[s:s+l,'OT'] += df['OT'].std() * 3
    labels[s:s+l] = 1

df_anom.to_csv(f'{PROJECT_DIR}/data/ETTh1_anomaly.csv', index=False)
np.save(f'{PROJECT_DIR}/data/labels.npy', labels)
print(f'Anomaly rate: {labels.mean()*100:.1f}%')


Anomaly rate: 4.1%


In [9]:
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import torch

class ETTDataset(Dataset):
    def __init__(self, csv_path, labels_path, split='train', seq_len=336, pred_len=96):
        df     = pd.read_csv(csv_path)
        labels = np.load(labels_path)
        data   = df.drop(columns=['date']).values.astype(np.float32)
        n      = len(data)
        cuts   = {'train':(0,int(.7*n)),'val':(int(.7*n),int(.8*n)),'test':(int(.8*n),n)}
        s,e    = cuts[split]
        self.scaler = StandardScaler()
        self.scaler.fit(data[cuts['train'][0]:cuts['train'][1]])
        data   = self.scaler.transform(data)
        self.data,self.labels = data[s:e],labels[s:e]
        self.seq_len,self.pred_len = seq_len,pred_len
    def __len__(self): return len(self.data)-self.seq_len-self.pred_len+1
    def __getitem__(self,i):
        x=self.data[i:i+self.seq_len]
        y=self.data[i+self.seq_len:i+self.seq_len+self.pred_len]
        return torch.tensor(x),torch.tensor(y),int(self.labels[i+self.seq_len-1])

train_ds = ETTDataset(f'{PROJECT_DIR}/data/ETTh1_anomaly.csv',
                       f'{PROJECT_DIR}/data/labels.npy','train')
val_ds   = ETTDataset(f'{PROJECT_DIR}/data/ETTh1_anomaly.csv',
                       f'{PROJECT_DIR}/data/labels.npy','val')
train_loader = DataLoader(train_ds,batch_size=32,shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,  batch_size=32,shuffle=False,num_workers=2)
print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} batches')


Train: 368 batches | Val: 41 batches


In [10]:
import torch.nn as nn, torch.nn.functional as F
from einops import rearrange

class PatchEmbedding(nn.Module):
    def __init__(self,patch_len,stride,d_model,seq_len):
        super().__init__()
        self.patch_len=patch_len; self.stride=stride
        self.num_patches=(seq_len-patch_len)//stride+1
        self.proj=nn.Linear(patch_len,d_model)
    def forward(self,x):          # x:(B,T,C)
        B,T,C=x.shape
        x=x.permute(0,2,1).unfold(-1,self.patch_len,self.stride)  # (B,C,N,P)
        x=rearrange(x,'b c n p->(b c) n p')
        return self.proj(x)        # (B*C,N,d_model)

class TransformerLayer(nn.Module):
    def __init__(self,d_model,n_heads,d_ff,dropout=0.1):
        super().__init__()
        self.attn =nn.MultiheadAttention(d_model,n_heads,dropout=dropout,batch_first=True)
        self.ff1  =nn.Linear(d_model,d_ff); self.ff2=nn.Linear(d_ff,d_model)
        self.norm1=nn.LayerNorm(d_model);   self.norm2=nn.LayerNorm(d_model)
        self.drop =nn.Dropout(dropout)
    def forward(self,x):
        a,_=self.attn(x,x,x)
        x=self.norm1(x+self.drop(a))
        return self.norm2(x+self.drop(self.ff2(self.drop(F.gelu(self.ff1(x))))))

class PatchTST(nn.Module):
    def __init__(self,seq_len=336,pred_len=96,n_channels=7,
                 patch_len=16,stride=8,d_model=64,n_heads=4,n_layers=2,d_ff=128,dropout=0.1):
        super().__init__()
        self.n_channels=n_channels; self.pred_len=pred_len
        self.patch_emb=PatchEmbedding(patch_len,stride,d_model,seq_len)
        N=self.patch_emb.num_patches
        self.encoder=nn.Sequential(*[TransformerLayer(d_model,n_heads,d_ff,dropout)
                                     for _ in range(n_layers)])
        self.head=nn.Linear(d_model*N,pred_len)
        self.pos=nn.Parameter(torch.zeros(1,N,d_model))
        nn.init.trunc_normal_(self.pos,std=0.02)
    def forward(self,x):   # x:(B,T,C)
        B=x.shape[0]
        z=self.patch_emb(x)+self.pos
        z=self.encoder(z)
        out=self.head(z.flatten(1))
        return rearrange(out,'(b c) p->b p c',b=B,c=self.n_channels)

model=PatchTST().cuda()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


Parameters: 322,656


In [11]:
import torch.optim as optim
CKPT=f'{PROJECT_DIR}/checkpoints/best.pt'
criterion=nn.MSELoss()
optimizer=optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)
scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=30)
best_val=float('inf')

for epoch in range(30):
    model.train(); tl=0
    for x,y,_ in train_loader:
        x,y=x.cuda(),y.cuda()
        optimizer.zero_grad()
        loss=criterion(model(x),y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step(); tl+=loss.item()
    scheduler.step()
    model.eval(); vl=0
    with torch.no_grad():
        for x,y,_ in val_loader: vl+=criterion(model(x.cuda()),y.cuda()).item()
    tl/=len(train_loader); vl/=len(val_loader)
    print(f'Epoch {epoch+1:02d}: train={tl:.4f} val={vl:.4f}')
    if vl<best_val:
        best_val=vl; torch.save(model.state_dict(),CKPT)
        print(f'  -> saved (val={vl:.4f})')


Epoch 01: train=0.4205 val=0.4458
  -> saved (val=0.4458)
Epoch 02: train=0.3630 val=0.4594
Epoch 03: train=0.3442 val=0.4891
Epoch 04: train=0.3296 val=0.4530
Epoch 05: train=0.3187 val=0.5120
Epoch 06: train=0.3092 val=0.4806
Epoch 07: train=0.3007 val=0.4883
Epoch 08: train=0.2917 val=0.5022
Epoch 09: train=0.2842 val=0.5290
Epoch 10: train=0.2774 val=0.5166
Epoch 11: train=0.2704 val=0.5440
Epoch 12: train=0.2662 val=0.5757
Epoch 13: train=0.2598 val=0.5219
Epoch 14: train=0.2552 val=0.5126
Epoch 15: train=0.2504 val=0.5309
Epoch 16: train=0.2456 val=0.5539
Epoch 17: train=0.2412 val=0.5509
Epoch 18: train=0.2368 val=0.5362
Epoch 19: train=0.2329 val=0.5442
Epoch 20: train=0.2295 val=0.5608
Epoch 21: train=0.2264 val=0.5481
Epoch 22: train=0.2239 val=0.5613
Epoch 23: train=0.2218 val=0.5730
Epoch 24: train=0.2198 val=0.5588
Epoch 25: train=0.2181 val=0.5688
Epoch 26: train=0.2165 val=0.5685
Epoch 27: train=0.2159 val=0.5702
Epoch 28: train=0.2148 val=0.5622
Epoch 29: train=0.2145 v

In [12]:
from sklearn.metrics import classification_report
model.load_state_dict(torch.load(CKPT)); model.eval()
test_ds=ETTDataset(f'{PROJECT_DIR}/data/ETTh1_anomaly.csv',
                    f'{PROJECT_DIR}/data/labels.npy','test')
test_loader=DataLoader(test_ds,batch_size=32,shuffle=False)
errors,true_labels=[],[]
with torch.no_grad():
    for x,y,lbl in test_loader:
        err=((model(x.cuda())-y.cuda())**2).mean(dim=(1,2))
        errors.extend(err.cpu().numpy()); true_labels.extend(lbl.numpy())
errors=np.array(errors); true_labels=np.array(true_labels)
threshold=np.percentile(errors,80)
print(classification_report(true_labels,(errors>threshold).astype(int)))
print(f'Threshold: {threshold:.4f}')


              precision    recall  f1-score   support

           0       1.00      0.80      0.89      3040
           1       0.00      0.15      0.01        13

    accuracy                           0.80      3053
   macro avg       0.50      0.48      0.45      3053
weighted avg       0.99      0.80      0.88      3053

Threshold: 0.5325


In [13]:
B,N,H,dk=32,41,4,16   # 32 batch, 41 patches, 4 heads, 16 head dim
Q=torch.randn(B*H,N,dk,device='cuda')
K=torch.randn(B*H,N,dk,device='cuda')
V=torch.randn(B*H,N,dk,device='cuda')

def naive_attn_pt(Q,K,V):
    s=Q.shape[-1]**-0.5
    return torch.matmul(torch.softmax(torch.matmul(Q,K.transpose(-2,-1))*s,dim=-1),V)

def time_fn(fn,*args,runs=200):
    for _ in range(20): fn(*args)
    torch.cuda.synchronize()
    s=torch.cuda.Event(enable_timing=True); e=torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(runs): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e)/runs

t_base=time_fn(naive_attn_pt,Q,K,V)
print(f'PyTorch attention baseline: {t_base:.3f} ms')
print(f'N^2 HBM estimate: {3*B*H*N*N*4/1e6:.3f} MB per pass')


PyTorch attention baseline: 0.081 ms
N^2 HBM estimate: 2.582 MB per pass


In [14]:
!pip install ninja
import importlib, torch.utils.cpp_extension as ce
importlib.reload(ce)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 7.4 MB/s eta 0:00:00


<module 'torch.utils.cpp_extension' from '/usr/local/lib/python3.12/dist-packages/torch/utils/cpp_extension.py'>

In [15]:
!apt-get install -y ninja-build

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  ninja-build
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 111 kB of archives.
After this operation, 358 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ninja-build amd64 1.10.1-1 [111 kB]
Fetched 111 kB in 1s (130 kB/s)
Selecting previously unselected package ninja-build.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.10.1-1_amd64.deb ...
Unpacking ninja-build (1.10.1-1) ...
Setting up ninja-build (1.10.1-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [16]:
!rm -rf /root/.cache/torch_extensions/py312_cu128/test_v1

In [27]:
test_src = r'''
#include <torch/extension.h>
__global__ void sk(float* x,float s,int n){int i=blockIdx.x*blockDim.x+threadIdx.x;if(i<n)x[i]*=s;}
torch::Tensor scale(torch::Tensor x,float s){auto o=x.clone();sk<<<(o.numel()+255)/256,256>>>(o.data_ptr<float>(),s,o.numel());return o;}
PYBIND11_MODULE(TORCH_EXTENSION_NAME,m){m.def("scale",&scale,"s");}
'''
with open('/tmp/test_ext.cu','w') as f: f.write(test_src)
ext = load(name='test_v1', sources=['/tmp/test_ext.cu'], verbose=True)
print(ext.scale(torch.ones(4,device='cuda'), 3.0))

tensor([3., 3., 3., 3.], device='cuda:0')


In [29]:
from torch.utils.cpp_extension import load

naive_src = r'''
#include <torch/extension.h>
#include <cuda_runtime.h>
#include <math.h>
__global__ void qk_k(const float* Q,const float* K,float* S,int N,int dk,float sc){
    int b=blockIdx.z,row=blockIdx.y*16+threadIdx.y,col=blockIdx.x*16+threadIdx.x;
    if(row>=N||col>=N) return;
    float a=0; for(int k=0;k<dk;k++) a+=Q[b*N*dk+row*dk+k]*K[b*N*dk+col*dk+k];
    S[b*N*N+row*N+col]=a*sc;
}
__global__ void sm_k(float* S,int B,int N){
    int b=blockIdx.y,row=blockIdx.x*256+threadIdx.x;
    if(row>=N) return;
    float* r=S+b*N*N+row*N;
    float mx=r[0]; for(int i=1;i<N;i++) mx=fmaxf(mx,r[i]);
    float s=0; for(int i=0;i<N;i++){r[i]=expf(r[i]-mx);s+=r[i];}
    for(int i=0;i<N;i++) r[i]/=s;
}
__global__ void pv_k(const float* P,const float* V,float* O,int N,int dk){
    int b=blockIdx.z,row=blockIdx.y*16+threadIdx.y,col=blockIdx.x*16+threadIdx.x;
    if(row>=N||col>=dk) return;
    float a=0; for(int k=0;k<N;k++) a+=P[b*N*N+row*N+k]*V[b*N*dk+k*dk+col];
    O[b*N*dk+row*dk+col]=a;
}
torch::Tensor naive_attention(torch::Tensor Q,torch::Tensor K,torch::Tensor V){
    int B=Q.size(0),N=Q.size(1),dk=Q.size(2);
    float sc=1.0f/sqrtf((float)dk);
    auto S=torch::empty({B,N,N},Q.options());
    auto O=torch::empty({B,N,dk},Q.options());
    dim3 t2(16,16);
    qk_k<<<dim3((N+15)/16,(N+15)/16,B),t2>>>(Q.data_ptr<float>(),K.data_ptr<float>(),S.data_ptr<float>(),N,dk,sc);
    sm_k<<<dim3((N+255)/256,B),256>>>(S.data_ptr<float>(),B,N);
    pv_k<<<dim3((dk+15)/16,(N+15)/16,B),t2>>>(S.data_ptr<float>(),V.data_ptr<float>(),O.data_ptr<float>(),N,dk);
    return O;
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME,m){m.def("naive_attention",&naive_attention,"naive");}
'''

with open('/tmp/naive_attn.cu','w') as f: f.write(naive_src)
naive_ext = load(name='naive_v1', sources=['/tmp/naive_attn.cu'], verbose=True)
print("Compiled successfully")

Compiled successfully


In [30]:
# Verify correctness
O_ref=naive_attn_pt(Q,K,V)
O_cuda=naive_ext.naive_attention(Q,K,V)
err=(O_ref-O_cuda).abs().max().item()
print(f'Max error: {err:.2e}')   # must be < 1e-4, else indexing bug
print('PASS' if err<1e-4 else 'FAIL -- do not proceed until this passes')

# Benchmark
t_naive=time_fn(naive_ext.naive_attention,Q,K,V)
print(f'Naive CUDA: {t_naive:.3f} ms')    # RECORD THIS
print(f'PyTorch ref: {t_base:.3f} ms')
print(f'Ratio: {t_naive/t_base:.1f}x -- expected to be slower, cuBLAS is optimized')


Max error: 4.77e-07
PASS
Naive CUDA: 0.230 ms
PyTorch ref: 0.081 ms
Ratio: 2.8x -- expected to be slower, cuBLAS is optimized


In [39]:
tiled_src = r'''
#include <torch/extension.h>
#include <cuda_runtime.h>
#include <math.h>
#define TS 16

__global__ void tiled_k(const float* Q,const float* K,const float* V,float* O,
                         int N,int dk,float scale){
    int b=blockIdx.z, qi=blockIdx.x*TS+threadIdx.x;
    bool valid_q = (qi < N);

    float q[16]={};
    if(valid_q)
        for(int d=0;d<dk;d++) q[d]=Q[b*N*dk+qi*dk+d];

    float o[16]={}, m_prev=-1e9f, l_prev=0.0f;
    __shared__ float Ks[TS][16], Vs[TS][16];

    int nt=(N+TS-1)/TS;
    for(int t=0;t<nt;t++){
        // ALL threads participate in loading -- no early return above
        int ki=t*TS+threadIdx.x;
        if(ki<N){
            for(int d=0;d<dk;d++){
                Ks[threadIdx.x][d]=K[b*N*dk+ki*dk+d];
                Vs[threadIdx.x][d]=V[b*N*dk+ki*dk+d];
            }
        } else {
            for(int d=0;d<dk;d++){ Ks[threadIdx.x][d]=0; Vs[threadIdx.x][d]=0; }
        }
        __syncthreads();

        if(valid_q){
            int te=min(TS,N-t*TS);
            float m_tile=-1e9f;
            for(int j=0;j<te;j++){
                float s=0;
                for(int d=0;d<dk;d++) s+=q[d]*Ks[j][d];
                s*=scale;
                m_tile=fmaxf(m_tile,s);
            }
            float m_new=fmaxf(m_prev,m_tile);
            float l_tile=0.0f;
            float o_tile[16]={};
            for(int j=0;j<te;j++){
                float s=0;
                for(int d=0;d<dk;d++) s+=q[d]*Ks[j][d];
                s*=scale;
                float es=expf(s-m_new);
                l_tile+=es;
                for(int d=0;d<dk;d++) o_tile[d]+=es*Vs[j][d];
            }
            float alpha=expf(m_prev-m_new);
            for(int d=0;d<dk;d++) o[d]=o[d]*alpha+o_tile[d];
            l_prev=l_prev*alpha+l_tile;
            m_prev=m_new;
        }
        __syncthreads();
    }

    if(valid_q)
        for(int d=0;d<dk;d++) O[b*N*dk+qi*dk+d]=o[d]/l_prev;
}

torch::Tensor tiled_attention(torch::Tensor Q,torch::Tensor K,torch::Tensor V){
    int B=Q.size(0),N=Q.size(1),dk=Q.size(2);
    auto O=torch::empty_like(Q);
    tiled_k<<<dim3((N+TS-1)/TS,1,B),TS>>>(
        Q.data_ptr<float>(),K.data_ptr<float>(),
        V.data_ptr<float>(),O.data_ptr<float>(),
        N,dk,1.0f/sqrtf((float)dk));
    return O;
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME,m){m.def("tiled_attention",&tiled_attention,"tiled");}
'''

with open('/tmp/tiled_attn4.cu','w') as f: f.write(tiled_src)
tiled_ext = load(name='tiled_v4', sources=['/tmp/tiled_attn4.cu'], verbose=True)
print("Compiled successfully")

Compiled successfully


In [41]:
O_tiled=tiled_ext.tiled_attention(Q,K,V)
err=(O_ref-O_tiled).abs().max().item()
print(f'Tiled error: {err:.2e}')
print('PASS' if err<1e-3 else 'FAIL')

t_tiled=time_fn(tiled_ext.tiled_attention,Q,K,V)
print(f'Naive CUDA: {t_naive:.3f} ms')
print(f'Tiled CUDA: {t_tiled:.3f} ms')
print(f'Speedup: {t_naive/t_tiled:.2f}x')


Tiled error: 4.77e-07
PASS
Naive CUDA: 0.230 ms
Tiled CUDA: 0.218 ms
Speedup: 1.05x


In [42]:
print(f"{'N':>6} {'Naive ms':>10} {'Tiled ms':>10} {'Speedup':>9} {'HBM ratio':>10}")
for Nt in [64,128,256,512,1024]:
    Qt=torch.randn(32,Nt,16,device='cuda')
    Kt=torch.randn(32,Nt,16,device='cuda')
    Vt=torch.randn(32,Nt,16,device='cuda')
    tn=time_fn(naive_ext.naive_attention,Qt,Kt,Vt,runs=100)
    tt=time_fn(tiled_ext.tiled_attention,Qt,Kt,Vt,runs=100)
    hn=3*32*Nt*Nt*4/1e6; ht=32*Nt*3*16*4/1e6
    print(f'{Nt:>6} {tn:>10.3f} {tt:>10.3f} {tn/tt:>9.2f}x {hn/ht:>10.1f}x')

     N   Naive ms   Tiled ms   Speedup  HBM ratio
    64      0.166      0.153      1.08x        4.0x
   128      0.609      0.401      1.52x        8.0x
   256      1.270      0.790      1.61x       16.0x
   512      4.584      3.111      1.47x       32.0x
  1024     19.483     12.523      1.56x       64.0x
